# Error Case 12 — 400 Bad Request: Malformed Request Body

**Error:** `400 Bad Request`  
**Root cause:** Wrong JSON schema — missing required fields or wrong types  
**Fix:** Check API spec, include all required fields

## Flow
```
POST /catalog/v1/{catalog}/namespaces/{ns}/tables
  → Polaris validates request body against schema
  → required field missing or wrong type
  → 400 Bad Request (before any RBAC check)
```

## Common mistakes
```
1. Table create: missing schema field
2. Catalog create: wrong storageType value
3. Grant: wrong privilege name
4. Namespace: namespace as string not array
```

In [8]:
import sys
sys.path.insert(0, '..')
import importlib
import polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [ ]:
# Cell 2: Get valid token first
r = get_token()
print_response(r, 'Valid token response')
valid_token = r.json()['access_token']
print(f'\n✅ Got valid token')

In [9]:
# Cell 2: Setup — catalog + namespace
r1 = create_catalog()
assert r1.status_code == 201, r1.text
r2 = create_namespace()
assert r2.status_code in [200, 201], r2.text
print('Setup: catalog ✅ | namespace ✅')

Setup: catalog ✅ | namespace ✅


In [10]:
# Cell 3a: 400 — namespace as string instead of array
r = requests.post(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(),
    json={
        'namespace': TEST_NAMESPACE,    # ← string, should be [TEST_NAMESPACE]
        'properties': {}
    }
)
print_response(r, '400 Namespace as string (not array)')
print(f'Expected: 400, Got: {r.status_code}')


400 Namespace as string (not array):
  Status:     400
  Request-Id: 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000111
  Body: {
  "error": {
    "message": "HTTP 400 Bad Request",
    "type": "WebApplicationException",
    "code": 400
  }
}
Expected: 400, Got: 400


In [11]:
# Cell 3b: 400 — table create missing required schema
r = requests.post(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NAMESPACE}/tables',
    headers=h(),
    json={
        'name': TEST_TABLE,
        # schema field missing entirely
    }
)
print_response(r, '400 Missing schema field')
print(f'Expected: 400, Got: {r.status_code}')


400 Missing schema field:
  Status:     400
  Request-Id: 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000113
  Body: {
  "error": {
    "message": "Invalid schema: null",
    "type": "IllegalArgumentException",
    "code": 400
  }
}
Expected: 400, Got: 400


In [12]:
# Cell 3c: 400 — wrong storageType value
r = requests.post(
    f'{BASE_MGMT}/catalogs',
    headers=h(),
    json={
        'catalog': {
            'name': 'bad-catalog',
            'type': 'INTERNAL',
            'properties': {
                'default-base-location': 's3a://bucket/bad/'
            },
            'storageConfigInfo': {
                'storageType': 'INVALID_TYPE',    # ← wrong value
                'allowedLocations': ['s3a://bucket/']
            }
        }
    }
)
print_response(r, '400 Wrong storageType')
print(f'Expected: 400, Got: {r.status_code}')


400 Wrong storageType:
  Status:     400
  Request-Id: 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000115
  Body: 
Expected: 400, Got: 400


In [13]:
# Cell 4: Find error logs
wait_for_logs(5)
print('Searching for 400 errors (last 1 min)...')
hits = search_400_errors(minutes_ago=1)
print_logs(hits)

⏳ Waiting 5s for logs to reach OpenSearch...
Searching for 400 errors (last 1 min)...
Found 5 log entries:

ℹ️  [2026-06-02T06:05:53.311Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000115
   realmId   : POLARIS
   message   : 192.168.194.1 - root [02/Jun/2026:06:05:53 +0000] "POST /api/management/v1/catalogs HTTP/1.1" 400 -

ℹ️  [2026-06-02T06:05:52.665Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000113
   realmId   : POLARIS
   message   : 192.168.194.1 - root [02/Jun/2026:06:05:52 +0000] "POST /api/catalog/v1/test-error-catalog/namespaces/test-ns/tables HTTP/1.1" 400 89

ℹ️  [2026-06-02T06:05:52.665Z] INFO
   logger    : org.apache.polaris.service.exception.IcebergExceptionMapper
   requestId : 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000113
   realmId   : POLARIS
   message   : Handling runtimeException Invalid schema: null

ℹ️  [2026-0

In [14]:
# Cell 5: Fix — correct request body
r_fix = requests.post(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NAMESPACE}/tables',
    headers=h(),
    json={
        'name': TEST_TABLE,
        'schema': {              # ← required field included
            'type': 'struct',
            'fields': [
                {'id': 1, 'name': 'id',   'required': True,  'type': 'long'},
                {'id': 2, 'name': 'data', 'required': False, 'type': 'string'}
            ]
        },
        'write-order': {'type': 'unsorted'},
        'stage-create': False
    }
)
print_response(r_fix, 'Correct table create')
print(f'Result: {r_fix.status_code}')

cleanup()
print('\n=== Summary ===')
print('Error:    400 Bad Request')
print('Cause:    Wrong JSON schema — missing/wrong fields')
print('Common:   namespace as string not array')
print('Common:   missing schema on table create')
print('Common:   wrong storageType value')
print('Fix:      Check Polaris REST API spec')
print('Note:     400 happens BEFORE RBAC — even admins get 400')


Correct table create:
  Status:     400
  Request-Id: 9887a791-517e-47dd-b249-e2806aed84a9_0000000000000000117
  Body: {
  "error": {
    "message": "HTTP 400 Bad Request",
    "type": "WebApplicationException",
    "code": 400
  }
}
Result: 400
🧹 Cleaning up test resources...
✅ Cleanup done

=== Summary ===
Error:    400 Bad Request
Cause:    Wrong JSON schema — missing/wrong fields
Common:   namespace as string not array
Common:   missing schema on table create
Common:   wrong storageType value
Fix:      Check Polaris REST API spec
Note:     400 happens BEFORE RBAC — even admins get 400
